# 리포트 58 — 자유공간 형상에서 문턱을 다시 재니 세 밴드가 SNR90 하나를 공유한다

> ### 한 일
> **자유공간 형상에서 경험 오경보율을 목표값에 고정해 문턱을 다시 잡고, 그 문턱 하나를 세 밴드 solve 가 공유할 때의 대가를 수치로 냈다.**

### 결과
1. 세 밴드의 solve 는 WiFi 에서 잰 문턱 SNR90 = 11.86 dB ⟨outputs/report05_derived.json : threshold.snr90_shared_db⟩ 하나를 공유한다.
2. LTE 가 자기 문턱을 쓰면 11.91 dB ⟨outputs/report05_derived.json : threshold.l1_own_snr90_db⟩ 로 +0.047 dB ⟨outputs/report05_derived.json : threshold.l1_delta_db⟩ 어긋나고, R90 은 -0.27% ⟨outputs/report05_derived.json : threshold.l1_range_shift_pct⟩ 움직인다.
3. 5G 의 요구 명목 Pfa 는 경험값의 18 배 ⟨outputs/report13_freespace.json : threshold.pfa.G1.ratio_emp_over_nominal → 역수⟩ 다 — 프레임 5 ⟨outputs/report13_freespace.json : waveforms.G1.M⟩개짜리 도플러 축이 그만큼 좁다.
4. 5G 의 dopoff 격자 5 ⟨outputs/report05_derived.json : threshold.g1_total_cells⟩칸은 M=5 ⟨outputs/report05_derived.json : threshold.g1_M⟩ 의 도플러 축 밖에 떨어져 자기 문턱을 못 세운다 — 그 행을 다음 단계로 넘긴다.

### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 문턱 | 자유공간 형상에서 경험 Pfa 를 목표 1e-04 ⟨outputs/report13_freespace.json : threshold.pfa.W1.target⟩ 에 고정하고 그때 요구되는 명목 Pfa 를 기록한다 — `src/freespace_detect.py:711` |
| 형상 의존 | 거리창·오버샘플·가드 규약이 챔버와 다르므로 형상마다 다시 잰다 — 챔버 형상의 배율은 [편 53 «경험 Pfa 교정»](53_cfar-calib.ipynb) 이 든다 |
| 공유 | stage_threshold 는 모드 목록의 첫 모드에서 SNR90 을 뽑아 세 밴드 solve 전부에 넘긴다 — `src/experiment_freespace_range.py:856` |

### 재현

```bash
for D in mini5pro mavic4pro matrice4e phantom4 s1000plus; do PYTHONPATH=src ~/.venvs/py312/bin/python src/experiment_freespace_range.py --stage all --mode W1,L1,G1 --drone $D; done
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python benchmark/verify_freespace.py
PYTHONPATH=src ~/.venvs/py312/bin/python src/build_part10_results.py
```

| | |
|---|---|
| 출력 | `outputs/report13_freespace.json`, `outputs/verify_freespace.json`, `outputs/report05_derived.json` |
| 소요 | 기하·규약 게이트 0.34 s ⟨outputs/report05_derived.json : runtime.verify_freespace_s⟩ |

### 앞 편에서

| 어디서 | 무엇을 알고 와야 하나 |
|---|---|
| [편 53 «경험 Pfa 교정»](53_cfar-calib.ipynb) | 명목 Pfa 를 경험 Pfa 로 교정하는 절차와 그 배율이 형상마다 다른 이유 |

---

## 세 파형 벤치마크 — 문턱을 경험 Pfa 로 교정한다

세 조명원은 각 표준이 늘 켜 두는 기준신호다 — WiFi VHT-LTF(W1) · LTE CRS(L1) · 5G SSB(G1). 제원은 [편 44 «상시 기준신호»](44_illuminators.ipynb) 가 들고, 여기서는 그 셋을 같은 검출기에 물린다.

경험 Pfa 를 목표 1e-04 ⟨outputs/report13_freespace.json : threshold.pfa.W1.target⟩ 에 고정하고, 그때 요구되는 명목 Pfa 를 기록한다(`src/freespace_detect.py:711`). 자유공간 형상은 거리창 · 오버샘플 · 가드 규약이 챔버와 다르므로 형상마다 다시 잰다.

| 모드 | 요구 명목 Pfa | 경험 Pfa | 경험/명목 |
|---|---|---|---|
| WiFi | 8.65e-05 ⟨outputs/report13_freespace.json : threshold.pfa.W1.nominal⟩ | 1.01e-04 ⟨outputs/report13_freespace.json : threshold.pfa.W1.empirical⟩ | 1.163 ⟨outputs/report13_freespace.json : threshold.pfa.W1.ratio_emp_over_nominal⟩ |
| LTE | 7.48e-05 ⟨outputs/report13_freespace.json : threshold.pfa.L1.nominal⟩ | 1.02e-04 ⟨outputs/report13_freespace.json : threshold.pfa.L1.empirical⟩ | 1.363 ⟨outputs/report13_freespace.json : threshold.pfa.L1.ratio_emp_over_nominal⟩ |
| 5G | 1.91e-03 ⟨outputs/report13_freespace.json : threshold.pfa.G1.nominal⟩ | 1.04e-04 ⟨outputs/report13_freespace.json : threshold.pfa.G1.empirical⟩ | 0.054 ⟨outputs/report13_freespace.json : threshold.pfa.G1.ratio_emp_over_nominal⟩ |

5G 의 요구 명목 Pfa 는 경험값의 18 배 ⟨outputs/report13_freespace.json : threshold.pfa.G1.ratio_emp_over_nominal → 역수⟩ 다 — 프레임 5 ⟨outputs/report13_freespace.json : waveforms.G1.M⟩개짜리 도플러 축이 그만큼 좁다.

세 밴드의 solve 는 이 중 W1 에서 잰 문턱 SNR90 = 11.86 dB ⟨outputs/report05_derived.json : threshold.snr90_shared_db⟩ 하나를 공유한다(`src/experiment_freespace_range.py:856`). 그 선택의 크기는 이렇다.

| 모드 | 자기 문턱 SNR90 | 공유 문턱과의 차 | R90 에 주는 차 |
|---|---|---|---|
| WiFi | 11.86 dB ⟨outputs/report05_derived.json : threshold.snr90_shared_db⟩ | 기준 | 기준 |
| LTE | 11.91 dB ⟨outputs/report05_derived.json : threshold.l1_own_snr90_db⟩ | +0.047 dB ⟨outputs/report05_derived.json : threshold.l1_delta_db⟩ | -0.27% ⟨outputs/report05_derived.json : threshold.l1_range_shift_pct⟩ |
| 5G | dopoff 격자 5 ⟨outputs/report05_derived.json : threshold.g1_total_cells⟩칸이 M=5 ⟨outputs/report05_derived.json : threshold.g1_M⟩ 의 도플러 축 밖 | — | — (다음 단계 1행) |

![report05_pf6_detector](../outputs/figures/report05_pf6_detector.png)

**그림 1.** 교정된 오경보율 위에서 세 파형이 요구하는 SNR 은 몇 dB 인가?

## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| 5G 의 dopoff 격자를 M 인식으로 고쳐 Pd=0.9 문턱을 직접 잰다 | 5G 의 R90 이 자기 문턱 위에 서고, 세 밴드가 문턱을 공유하는 위 표의 행이 닫힌다 | `src/experiment_freespace_range.py:856` → [편 60 «R90 표»](60_r90.ipynb) |